In [8]:
import pandas as pd 
test = pd.read_csv("test.csv")
validation = pd.read_csv("validation.csv")

In [9]:
import  joblib 
transformer= joblib.load("transformer.pkl")

In [10]:
train = pd.read_csv("train_data_2.csv")

In [4]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'total_items',
 'total_price',
 'total_freight_value',
 'total_payment',
 'max_installments',
 'payment_count',
 'delivery_status',
 'customer_state',
 'seller_id',
 'seller_state',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng',
 'month_name',
 'day_name',
 'distance_km']

In [5]:
features =["total_items" ,"total_price" ,"total_freight_value",
                       "total_payment" ,"max_installments","payment_count",
                       "distance_km","seller_state" , "customer_state","month_name","day_name"]
target = "delivery_status"

In [11]:
X_train = train[features]
y_train = train[target]

X_validation = validation[features]
y_validation = validation[target]

X_test = test[features]
y_test = test[target]

In [12]:
X_train.shape, y_train.shape, X_validation.shape, y_validation.shape, X_test.shape, y_test.shape

((69608, 11), (69608,), (14916, 11), (14916,), (14917, 11), (14917,))

In [13]:
X_train_transformed = transformer.transform(X_train)
X_validation_transformed = transformer.transform(X_validation)
X_test_transformed = transformer.transform(X_test)

In [14]:
X_train_transformed.shape , X_validation_transformed.shape,X_test_transformed.shape

((69608, 77), (14916, 77), (14917, 77))

In [21]:
from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy = "most_frequent")
baseline.fit(X_train_transformed , y_train)


,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.12,0.88]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](2,)","['late','on time']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,77
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [22]:
baseline_prediction = baseline.predict(X_validation_transformed)


In [26]:
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
print("Accuracy:",accuracy_score(y_validation,baseline_prediction))
print("Precision:",precision_score(y_validation,
                                 baseline_prediction,
                                   pos_label='late',
                                   zero_division = 0
                                  ))
print("Recall:",recall_score(y_validation,
                             baseline_prediction,
                             pos_label='late',
                             zero_division = 0
                            ))
print("F1-score:",f1_score(y_validation,
                             baseline_prediction,
                             pos_label='late',
                             zero_division = 0
                          ))
                             

Accuracy: 0.9310807186913381
Precision: 0.0
Recall: 0.0
F1-score: 0.0


In [27]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(
    max_iter = 3000,
    class_weight = "balanced"
)
logistic_model.fit(X_train_transformed,y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [28]:
logistic_prediction = logistic_model.predict(X_validation_transformed) 

In [29]:
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
print("Accuracy:",accuracy_score(y_validation,logistic_prediction))
print("Precision:",precision_score(y_validation,
                                  logistic_prediction,
                                   pos_label='late',
                                   zero_division = 0
                                  ))
print("Recall:",recall_score(y_validation,
                             logistic_prediction,
                             pos_label='late',
                             zero_division = 0
                            ))
print("F1-score:",f1_score(y_validation,
                             logistic_prediction,
                             pos_label='late',
                             zero_division = 0
                          ))

Accuracy: 0.9050683829444891
Precision: 0.08008658008658008
Recall: 0.03599221789883268
F1-score: 0.049664429530201344


In [ ]:
from sklearn.ensemble import RandomForestClassifier 
rf_model = RandomForestClassifier(
    n_estimators = 200 ,
    class_weight = "balanced",
    random_state = 42,
    n_jobs = -1
)
rf_model.fit(X_train_transformed,y_train)

In [34]:
rf_model_prediction = rf_model.predict(X_validation_transformed)

In [35]:
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
print("Accuracy:",accuracy_score(y_validation,rf_model_prediction))
print("Precision:",precision_score(y_validation,
                                  logistic_prediction,
                                   pos_label='late',
                                   zero_division = 0
                                  ))
print("Recall:",recall_score(y_validation,
                             rf_model_prediction,
                             pos_label='late',
                             zero_division = 0
                            ))
print("F1-score:",f1_score(y_validation,
                             rf_model_prediction,
                             pos_label='late',
                             zero_division = 0
                          ))

Accuracy: 0.9288683293108072
Precision: 0.08008658008658008
Recall: 0.0048638132295719845
F1-score: 0.009337068160597572


In [37]:
from sklearn.metrics import confusion_matrix
print("Logistic Regerssion:")
print(confusion_matrix(y_validation , logistic_prediction))

print("\n Random Forest:")
print(confusion_matrix(y_validation , rf_model_prediction))

Logistic Regerssion:
[[   37   991]
 [  425 13463]]

 Random Forest:
[[    5  1023]
 [   38 13850]]


In [38]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10]
}

In [39]:
from sklearn.metrics import make_scorer, f1_score

late_f1 = make_scorer(
    f1_score,
    pos_label="late"
)

In [40]:
rf_tuning = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid,
    n_iter=10,
    scoring=late_f1,
    cv=3,
    random_state=42,
    n_jobs=-1
)
rf_tuning.fit(X_train_transformed, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 5, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",make_scorer(f...os_label=late)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple met

In [41]:
best_rf = rf_tuning.best_estimator_

rf_validation_prediction = best_rf.predict(X_validation_transformed)

In [42]:
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
print("Accuracy:",accuracy_score(y_validation,rf_model_prediction))
print("Precision:",precision_score(y_validation,
                                 rf_validation_prediction,
                                   pos_label='late',
                                  ))
print("Recall:",recall_score(y_validation,
                             rf_validation_prediction,
                             pos_label='late',
                            ))
print("F1-score:",f1_score(y_validation,
                             rf_validation_prediction,
                             pos_label='late',
                          ))

Accuracy: 0.9288683293108072
Precision: 0.07142857142857142
Recall: 0.011673151750972763
F1-score: 0.020066889632107024


In [43]:
logistic_prob = logistic_model.predict_proba(X_validation_transformed)[:, 0]


In [44]:
rf_prob = rf_tuning.best_estimator_.predict_proba(X_validation_transformed)[:, 0]

In [45]:
from sklearn.metrics import f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:
    
    logistic_pred = [
        "late" if p >= threshold else "on time"
        for p in logistic_prob
    ]
    
    print(
        "Threshold:", threshold,
        "F1:", f1_score(y_validation, logistic_pred, pos_label="late")
    )

Threshold: 0.2 F1: 0.12971055798340617
Threshold: 0.3 F1: 0.13215577782506915
Threshold: 0.4 F1: 0.09189001044204664
Threshold: 0.5 F1: 0.049664429530201344


In [46]:
for threshold in thresholds:
    
    rf_pred = [
        "late" if p >= threshold else "on time"
        for p in rf_prob
    ]
    
    print(
        "Threshold:", threshold,
        "F1:", f1_score(y_validation, rf_pred, pos_label="late")
    )

Threshold: 0.2 F1: 0.12895132965378825
Threshold: 0.3 F1: 0.12904845593773537
Threshold: 0.4 F1: 0.13327131061751366
Threshold: 0.5 F1: 0.020066889632107024


In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

best_threshold = 0.4

rf_best_pred = [
    "late" if p >= best_threshold else "on time"
    for p in rf_prob
]

print("Accuracy:", accuracy_score(y_validation, rf_best_pred))
print("Precision:", precision_score(y_validation, rf_best_pred, pos_label="late"))
print("Recall:", recall_score(y_validation, rf_best_pred, pos_label="late"))
print("F1-score:", f1_score(y_validation, rf_best_pred, pos_label="late"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, rf_best_pred, labels=["late", "on time"]))



Accuracy: 0.5003352105122016
Precision: 0.075683529256373
Recall: 0.5573929961089494
F1-score: 0.13327131061751366

Confusion Matrix:
[[ 573  455]
 [6998 6890]]


In [48]:
import joblib

final_model = rf_tuning.best_estimator_

joblib.dump(final_model, "final_random_forest.pkl")

['final_random_forest.pkl']

In [49]:
joblib.dump(0.4, "final_threshold.pkl")

['final_threshold.pkl']

In [50]:
final_model = joblib.load("final_random_forest.pkl")
final_threshold = joblib.load("final_threshold.pkl")

In [51]:
test_prob = final_model.predict_proba(X_test_transformed)[:, 0]

In [52]:
test_prediction = [
    "late" if p >= final_threshold else "on time"
    for p in test_prob
]

In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Final Test Results")
print("-------------------")

print("Accuracy:", accuracy_score(y_test, test_prediction))
print("Precision:", precision_score(y_test, test_prediction, pos_label="late"))
print("Recall:", recall_score(y_test, test_prediction, pos_label="late"))
print("F1-score:", f1_score(y_test, test_prediction, pos_label="late"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_prediction, labels=["late", "on time"]))

Final Test Results
-------------------
Accuracy: 0.6833143393443722
Precision: 0.0846676370693838
Recall: 0.2684615384615385
F1-score: 0.12873478421246773

Confusion Matrix:
[[ 349  951]
 [3773 9844]]


In [54]:
final_results = {
    "model": "Random Forest",
    "threshold": final_threshold,
    "accuracy": accuracy_score(y_test, test_prediction),
    "precision_late": precision_score(y_test, test_prediction, pos_label="late"),
    "recall_late": recall_score(y_test, test_prediction, pos_label="late"),
    "f1_late": f1_score(y_test, test_prediction, pos_label="late")
}

final_results

{'model': 'Random Forest',
 'threshold': 0.4,
 'accuracy': 0.6833143393443722,
 'precision_late': 0.0846676370693838,
 'recall_late': 0.2684615384615385,
 'f1_late': 0.12873478421246773}

In [55]:
joblib.dump(final_results, "final_results.pkl")

['final_results.pkl']